# Online Retail — quick EDA report

Self-contained demo on a synthetic mini-dataset that mimics the Online Retail II structure (invoice, stock_code, quantity, unit_price, customer_id, country, invoice_dt). The full pipeline in `src/build_warehouse.py` works on the real UCI dataset — see `data/README.md` for the download.

This report is just to show what the analysis looks like, without needing the 45MB Excel file.

In [ ]:
import sqlite3, tempfile, random
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

rng = np.random.default_rng(42)
random.seed(42)

# --- synthesise a small but plausible online-retail dataset ---
countries = ['United Kingdom']*70 + ['Germany']*6 + ['France']*5 + ['EIRE']*4 + ['Spain']*3 + ['Netherlands']*3 + ['Belgium']*2 + ['Australia']*2 + ['Switzerland']*2 + ['Portugal']*1 + ['Italy']*1 + ['Sweden']*1
products = [
    ('22423', 'REGENCY CAKESTAND 3 TIER', 12.75),
    ('85123A','WHITE HANGING HEART T-LIGHT HOLDER', 2.95),
    ('47566', 'PARTY BUNTING', 4.65),
    ('20725', 'LUNCH BAG RED RETROSPOT', 1.65),
    ('22720', 'SET OF 3 CAKE TINS PANTRY DESIGN', 4.95),
    ('21212', 'PACK OF 72 RETROSPOT CAKE CASES', 0.55),
    ('23298', 'SPOTTY BUNTING', 4.95),
    ('22086', 'PAPER CHAIN KIT 50\u2019S CHRISTMAS', 2.95),
    ('22910', 'PAPER CHAIN KIT VINTAGE CHRISTMAS', 2.95),
    ('21931', 'JUMBO STORAGE BAG SUKI', 1.95),
    ('22197', 'POPCORN HOLDER', 0.85),
    ('20727', 'LUNCH BAG BLACK SKULL', 1.65),
    ('21080', 'SET/20 RED RETROSPOT PAPER NAPKINS', 0.85),
    ('23203', 'JUMBO BAG VINTAGE DOILY', 1.95),
    ('22382', 'LUNCH BAG SPACEBOY DESIGN', 1.65),
]

n_rows = 8000
start = pd.Timestamp('2024-01-01')
end = pd.Timestamp('2024-12-31')
dates = pd.to_datetime(rng.integers(start.value//10**9, end.value//10**9, size=n_rows), unit='s')
rows = []
invoice_seed = 536000
current_invoice = invoice_seed
current_customer = 12345
remain = 0
for i in range(n_rows):
    if remain == 0:
        current_invoice += 1
        current_customer = int(12000 + rng.integers(0, 800))
        country = random.choice(countries)
        remain = int(rng.integers(1, 12))
    sc, desc, base_price = products[int(rng.integers(0, len(products)))]
    qty = int(rng.integers(1, 25))
    # 2% returns (negative qty)
    if rng.random() < 0.02:
        qty = -qty
    price = round(base_price * float(rng.normal(1.0, 0.05)), 2)
    rows.append((str(current_invoice) if qty>0 else 'C'+str(current_invoice), sc, desc, qty,
                 dates[i].strftime('%Y-%m-%d %H:%M:%S'), price, current_customer, country))
    remain -= 1
df = pd.DataFrame(rows, columns=['invoice','stock_code','description','quantity','invoice_dt','unit_price','customer_id','country'])
df['invoice_dt'] = pd.to_datetime(df['invoice_dt'])
df['revenue'] = df['quantity'] * df['unit_price']
print(f'rows: {len(df):,}  invoices: {df.invoice.nunique():,}  customers: {df.customer_id.nunique():,}  countries: {df.country.nunique()}')
df.head()

## load to sqlite + run analysis SQL

Same pattern as the real pipeline — just in-memory.

In [ ]:
con = sqlite3.connect(':memory:')
df.to_sql('sales', con, index=False)
con.execute('CREATE INDEX idx_sales_dt ON sales(invoice_dt)')
print(con.execute('SELECT COUNT(*) FROM sales').fetchone())

## monthly revenue trend

In [ ]:
monthly = pd.read_sql_query("""
SELECT strftime('%Y-%m', invoice_dt) AS month,
       ROUND(SUM(quantity * unit_price), 2) AS revenue_gbp
FROM sales
WHERE quantity > 0
GROUP BY 1
ORDER BY 1
""", con, parse_dates=['month'])
fig, ax = plt.subplots(figsize=(10, 4))
monthly.plot(x='month', y='revenue_gbp', ax=ax, legend=False, marker='o', color='#4C78A8')
ax.set_title('monthly revenue (GBP)')
ax.set_ylabel('revenue (GBP)')
ax.set_xlabel('')
ax.grid(alpha=0.3)
plt.tight_layout()

## top 10 products by revenue

In [ ]:
top = pd.read_sql_query("""
SELECT description,
       SUM(quantity) AS units_sold,
       ROUND(SUM(quantity * unit_price), 2) AS revenue_gbp
FROM sales
WHERE quantity > 0
GROUP BY 1
ORDER BY revenue_gbp DESC
LIMIT 10
""", con)
top

## non-UK markets

In [ ]:
country = pd.read_sql_query("""
SELECT country,
       ROUND(SUM(quantity * unit_price), 2) AS revenue_gbp,
       COUNT(DISTINCT invoice) AS invoices
FROM sales
WHERE quantity > 0 AND country <> 'United Kingdom'
GROUP BY 1
ORDER BY revenue_gbp DESC
""", con)
fig, ax = plt.subplots(figsize=(8, 5))
country.plot.barh(x='country', y='revenue_gbp', ax=ax, legend=False, color='#54A24B')
ax.invert_yaxis()
ax.set_title('top non-UK markets by revenue')
ax.set_xlabel('revenue (GBP)')
plt.tight_layout()
country

## return rate by country

In [ ]:
ret = pd.read_sql_query("""
SELECT country,
       SUM(CASE WHEN quantity < 0 THEN -quantity * unit_price ELSE 0 END) AS returns_gbp,
       SUM(CASE WHEN quantity > 0 THEN quantity * unit_price ELSE 0 END)   AS gross_gbp
FROM sales
GROUP BY 1
HAVING gross_gbp > 0
ORDER BY returns_gbp / gross_gbp DESC
""", con)
ret['return_rate_pct'] = (ret['returns_gbp'] / ret['gross_gbp'] * 100).round(2)
ret[['country', 'gross_gbp', 'returns_gbp', 'return_rate_pct']].head(10)

## simple RFM segmentation

recency = days since last purchase, frequency = distinct invoices, monetary = total spend.

In [ ]:
snapshot = df['invoice_dt'].max() + pd.Timedelta(days=1)
rfm = (df[df['quantity'] > 0]
       .groupby('customer_id')
       .agg(recency_days=('invoice_dt', lambda s: (snapshot - s.max()).days),
            frequency=('invoice', 'nunique'),
            monetary=('revenue', 'sum')))
rfm['R'] = pd.qcut(rfm['recency_days'].rank(method='first'), 4, labels=[4,3,2,1]).astype(int)
rfm['F'] = pd.qcut(rfm['frequency'].rank(method='first'),  4, labels=[1,2,3,4]).astype(int)
rfm['M'] = pd.qcut(rfm['monetary'].rank(method='first'),    4, labels=[1,2,3,4]).astype(int)
rfm['rfm_score'] = rfm['R'] + rfm['F'] + rfm['M']
fig, ax = plt.subplots(figsize=(8, 4))
rfm['rfm_score'].value_counts().sort_index().plot.bar(ax=ax, color='#F58518')
ax.set_title('RFM score distribution (3 = worst, 12 = best)')
ax.set_xlabel('R+F+M')
ax.set_ylabel('customers')
plt.tight_layout()
rfm.head(10)

---
*generated by `notebooks/eda_report.ipynb` — self-contained, no external data needed.*